# Gaussian Naive Bayes Classifier
Binary classification on MNIST: **digit 0** vs **Not 0**.

Uses Gaussian Naive Bayes with class-weighted priors and weighted likelihood estimation to handle the ~10% / 90% class imbalance.

In [30]:
import numpy as np

## Load Preprocessed Data

In [ ]:
from preprocessing import preprocess
# Preprocess the data with PCA features (Yielded better results than flatten features)
X_train, y_train, X_val, y_val, X_test, y_test, weights = preprocess(feature_method="pca", n_pca=50)


Loading MNIST dataset...
Split completed: Train=54000, Val=6000, Test=10000


## Model Definition

In [ ]:
class GaussianNaiveBayes:
    def __init__(self):
        self.means      = None   # shape: (n_classes, n_features = 50 after PCA)
        self.variances  = None   # shape: (n_classes, n_features = 50 after PCA)
        self.log_priors = None   # shape: (n_classes,)
        self.classes    = None   # array: [0, 1]

    def fit(self, X, y, class_weights=None):
        n_samples, n_features = X.shape
        self.classes = np.unique(y)
        n_classes = len(self.classes)

        # --- Log Priors ---
        # P(class) = count(class) / total, scaled by optional weight.
        # We store log-probabilities to avoid underflow when multiplying
        # many small likelihoods together later.
        self.log_priors = np.zeros(n_classes)
        for i, c in enumerate(self.classes):
            prior = np.sum(y == c) / n_samples
            if class_weights is not None:
                prior *= class_weights[c]        # up-weight minority class
            self.log_priors[i] = np.log(prior)

        # --- Per-class means and variances ---
        # Gaussian NB assumes features are conditionally independent given
        # the class, so we only need the per-feature mean and variance.
        self.means     = np.zeros((n_classes, n_features))
        self.variances = np.zeros((n_classes, n_features))

        for i, c in enumerate(self.classes):
            X_c = X[y == c]                      # samples belonging to class c

            if class_weights is not None:
                # Weighted statistics: each sample gets the weight of its class,
                # then we normalise so the weights sum to 1.
                w = np.full(len(X_c), class_weights[c])
                w = w / w.sum()
                self.means[i]     = (w[:, None] * X_c).sum(axis=0)
                self.variances[i] = (w[:, None] * (X_c - self.means[i]) ** 2).sum(axis=0)
            else:
                self.means[i]     = X_c.mean(axis=0)
                self.variances[i] = X_c.var(axis=0)

        # Laplacian smoothing: prevents log(0) when a feature has zero
        # variance (e.g. a pixel that is always black across a class).
        self.variances += 1e-9

    def _log_likelihood(self, X, class_idx):
        # Gaussian log-PDF: log P(x | class) = sum over features of
        #   -0.5 * log(2π * σ²)  -  (x - μ)² / (2σ²)
        # Summing the log-PDF across features exploits the conditional
        # independence assumption (products become sums in log-space).
        mean = self.means[class_idx]
        var  = self.variances[class_idx]

        log_norm  = -0.5 * np.log(2 * np.pi * var)           # normalisation term
        log_gauss = -0.5 * ((X - mean) ** 2) / var           # exponent term
        return (log_norm + log_gauss).sum(axis=1)             # shape: (n_samples,)

    def predict(self, X):
        # Score each class: log P(class | x) ∝ log P(class) + log P(x | class)
        # We pick the class with the highest (unnormalised) log-posterior.
        log_posteriors = np.array([
            self.log_priors[i] + self._log_likelihood(X, i)
            for i in range(len(self.classes))
        ])  # shape: (n_classes, n_samples)

        best_class_indices = np.argmax(log_posteriors, axis=0)
        return self.classes[best_class_indices]

    def evaluate(self, X, y, dataset_name="Validation"):
        y_pred = self.predict(X)
        y_true = y

        # Confusion matrix building blocks
        # Positive class = 0 (the digit zero)
        TP = np.sum((y_pred == 0) & (y_true == 0))
        TN = np.sum((y_pred == 1) & (y_true == 1))
        FP = np.sum((y_pred == 0) & (y_true == 1))
        FN = np.sum((y_pred == 1) & (y_true == 0))

        accuracy  = (TP + TN) / (TP + TN + FP + FN)
        precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
        recall    = TP / (TP + FN) if (TP + FN) > 0 else 0.0
        f1        = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

        conf_matrix = np.array([[TP, FN],
                                 [FP, TN]])

        print(f"--- {dataset_name} Results ---")
        print(f"Accuracy  : {accuracy:.4f}")
        print(f"Precision : {precision:.4f}")
        print(f"Recall    : {recall:.4f}")
        print(f"F1-Score  : {f1:.4f}")
        print(f"Confusion Matrix:")
        print(f"                 Predicted 0   Predicted 1")
        print(f"  Actual 0   :   {TP:<12}  {FN}")
        print(f"  Actual 1   :   {FP:<12}  {TN}")

## Training

In [33]:
model = GaussianNaiveBayes()
model.fit(X_train, y_train, class_weights=weights)

## Evaluation on Validation Set

In [34]:
model.evaluate(X_val, y_val, dataset_name="Validation")

--- Validation Results ---
Accuracy  : 0.9783
Precision : 0.8499
Recall    : 0.9455
F1-Score  : 0.8952
Confusion Matrix:
                 Predicted 0   Predicted 1
  Actual 0   :   555           32
  Actual 1   :   98            5315


## Evaluation on Test Set

In [35]:
model.evaluate(X_test, y_test, dataset_name="Test")

--- Test Results ---
Accuracy  : 0.9756
Precision : 0.8167
Recall    : 0.9684
F1-Score  : 0.8861
Confusion Matrix:
                 Predicted 0   Predicted 1
  Actual 0   :   949           31
  Actual 1   :   213           8807


## Experiment: Flatten vs PCA vs HOG

In [ ]:
# Load flattened and HOG features for comparison
X_train_flatten, y_train_flatten, X_val_flatten, y_val_flatten, X_test_flatten, y_test_flatten, weights_flatten = preprocess(feature_method="flatten")
X_train_HOG, y_train_HOG, X_val_HOG, y_val_HOG, X_test_HOG, y_test_HOG, weights_HOG = preprocess(feature_method="hog")

# Train and evaluate flattened pixel features
model_flatten = GaussianNaiveBayes()
model_flatten.fit(X_train_flatten, y_train_flatten, class_weights=weights_flatten)
model_flatten.evaluate(X_val_flatten, y_val_flatten, dataset_name="Validation (Flattened)")

# Train and evaluate HOG features
model_HOG = GaussianNaiveBayes()
model_HOG.fit(X_train_HOG, y_train_HOG, class_weights=weights_HOG)
model_HOG.evaluate(X_val_HOG, y_val_HOG, dataset_name="Validation (HOG)")